In [1]:
import numpy as np

In [2]:
scores = np.array([
    [0.8, 0.3, 0.5, 0.2],
    [0.4, 0.9, 0.1, 0.7],
    [0.2, 0.6, 0.8, 0.3],
    [0.1, 0.4, 0.6, 0.9]
],dtype=float)

In [8]:
# Casual Mask 
#np.triu return upper trianggle k = 1  means exclude diagonal 
mask = np.triu(np.ones((4,4)), k=1)
mask

array([[0., 1., 1., 1.],
       [0., 0., 1., 1.],
       [0., 0., 0., 1.],
       [0., 0., 0., 0.]])

In [4]:
scores[mask== 1] = -np.inf

In [5]:
print("Masked scores:")
print(scores)
print()


Masked scores:
[[ 0.8 -inf -inf -inf]
 [ 0.4  0.9 -inf -inf]
 [ 0.2  0.6  0.8 -inf]
 [ 0.1  0.4  0.6  0.9]]



In [6]:
def softmax(x):
    # WHY subtract max: numerical stability, prevents e^large = overflow
    e = np.exp(x - np.max(x))
    return e / e.sum()

In [7]:
attention_weights = np.array([softmax(row) for row in scores])
print("Attention weights after softmax:")
print(np.round(attention_weights, 3))

Attention weights after softmax:
[[1.    0.    0.    0.   ]
 [0.378 0.622 0.    0.   ]
 [0.232 0.346 0.422 0.   ]
 [0.161 0.217 0.265 0.358]]


# Scaled Dot-Product Attention from Scratch

In [18]:
def scaled_dot_product_attention(Q, K, V, mask=True):
     # Q = Query, K = Key, V = Value
    # WHY three matrices: each token asks a question (Q), 
    # broadcasts what it contains (K), and holds its actual value (V)

    d_k = Q.shape[-1]
    # compute raw attention score 

    scores = Q @ K.T

    # scale by d_k
    scores = scores / np.sqrt(d_k)

    # causal masking 
    if mask:
        seq_len = Q.shape[0]
        casual_mask = causal_mask = np.triu(np.ones((seq_len, seq_len)), k=1)
        scores[causal_mask == 1] = -np.inf

    # Softmax -> attention weights 
    weights = np.array([softmax(row) for row in scores])

    # Weighted sum of values 
    output= weights @ V

    return output, weights

In [19]:
np.random.seed(42)
seq_len, d_k = 4, 8
Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)

In [20]:
V = np.random.randn(seq_len, d_k)

In [21]:
output, weights = scaled_dot_product_attention(Q, K, V, mask=True)

In [22]:
print("Attention weights:")
print(np.round(weights, 3))
print()
print("Output shape:", output.shape)
print("Output (first token):", np.round(output[0], 3))

Attention weights:
[[1.    0.    0.    0.   ]
 [0.828 0.172 0.    0.   ]
 [0.702 0.131 0.166 0.   ]
 [0.178 0.492 0.201 0.13 ]]

Output shape: (4, 8)
Output (first token): [ 0.813  1.356 -0.072  1.004  0.362 -0.645  0.361  1.538]


# multi_head_attention

In [24]:
def multi_head_attention(X, num_heads, mask=True):
    seq_len, d_model = X.shape
    
    # WHY d_model must be divisible by num_heads
    # each head gets an equal slice of the model dimension
    assert d_model % num_heads == 0
    d_k = d_model // num_heads
    
    np.random.seed(42)
    
    
    # WHY separate projections: lets each head learn different relationships
    heads = []
    for i in range(num_heads):
        W_Q = np.random.randn(d_model, d_k)
        W_K = np.random.randn(d_model, d_k)
        W_V = np.random.randn(d_model, d_k)
        
        Q = X @ W_Q  # (seq_len, d_k)
        K = X @ W_K
        V = X @ W_V
        
        head_output, _ = scaled_dot_product_attention(Q, K, V, mask=mask)
        heads.append(head_output)
    
    
    concat = np.concatenate(heads, axis=-1)  # (seq_len, d_model)
    
    # Final projection to mix head outputs together
    W_O = np.random.randn(d_model, d_model)
    output = concat @ W_O
    
    return output

# 4 tokens, 8 dimensional embeddings, 2 heads
X = np.random.randn(4, 8)
output = multi_head_attention(X, num_heads=2, mask=True)

print("Input shape:", X.shape)
print("Output shape:", output.shape)
print("Head dimension (d_k):", 8 // 2)

Input shape: (4, 8)
Output shape: (4, 8)
Head dimension (d_k): 4


## Day 57 — LLM Architecture from Scratch

### What I built
- Causal attention mask from scratch in NumPy
- Scaled dot-product attention (Q, K, V) from scratch
- Multi-head attention from scratch

### Core concepts
- BERT = encoder-only, bidirectional attention, full matrix
- GPT = decoder-only, causal (masked) attention, lower triangular
- Causal mask sets future positions to -inf → 0 after softmax
- Scaling by sqrt(d_k) prevents softmax saturation and gradient vanishing
- Output vector = weighted sum of Value vectors = enriched token representation
- Multi-head: each head specializes, concatenated then projected back to d_model

### Key numbers
- GPT-2 small: 12 layers, 12 heads, d_model=768, d_k=64 per head
- Q,K,V projections alone: ~21M of 117M total parameters

### Critical rules
- Later tokens have richer representations — they've seen more context
- Input shape = output shape through every transformer layer
- num_heads must divide d_model exactly
- Causal mask is what makes next-token prediction training valid